# 第 2 章 · 本体与运动学：关节角怎么变成手的位置

配套讲义：仓库里 `robot-system-architecture/notes/lessons/02-kinematics.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

（连杆长度 L1 = 0.5 m、L2 = 0.4 m 和角度都是编的）

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第 3 步 FK | $x = L_1c_1 + L_2c_{12}$，$y = L_1s_1 + L_2s_{12}$ | `fk(theta)` | θ = (30°, 60°) → (0.4330, 0.6500) m |
| 工作空间 | 内圈 $\lvert L_1-L_2\rvert$、外圈 $L_1+L_2$ | — | 0.1 m、0.9 m |
| 第 4 步 IK 两组解 | 阻尼最小二乘 | `dls_ik(target, theta0)` | (30°, 60°) 和 (82.66°, −60°) 都到同一点 |
| 第 5 步 雅可比 | $\dot x = J\dot\theta$ | `jacobian` vs `numeric_jacobian` | 解析式和差分一致 |
| 第 5 步 静力学 | $\tau = J^\top F$ | `jacobian(theta).T @ F` | 挂 10 N → τ = (−4.33, 0) N·m |
| 第 6 步 奇异 | $\det J = L_1L_2\sin\theta_2$ | `np.linalg.det`、`manipulability` | θ2 = 60° → 0.1732；θ2 = 0 → 0 |

连杆长度和角度都是编的（L1 = 0.5 m，L2 = 0.4 m），不是练习题里的数。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
L1, L2 = 0.5, 0.4  # 两节连杆长度（m），编的

**函数 `fk`**：两连杆正运动学：x = L1 cos θ1 + L2 cos(θ1+θ2)，y = L1 sin θ1 + L2 sin(θ1+θ2)。

In [ ]:
def fk(theta: np.ndarray) -> np.ndarray:
    """两连杆正运动学：x = L1 cos θ1 + L2 cos(θ1+θ2)，y = L1 sin θ1 + L2 sin(θ1+θ2)。"""
    t1, t2 = theta
    return np.array([L1 * np.cos(t1) + L2 * np.cos(t1 + t2), L1 * np.sin(t1) + L2 * np.sin(t1 + t2)])

**函数 `jacobian`**：两连杆雅可比 J(θ)：第 j 列 = 只转第 j 个关节时手的速度方向。

In [ ]:
def jacobian(theta: np.ndarray) -> np.ndarray:
    """两连杆雅可比 J(θ)：第 j 列 = 只转第 j 个关节时手的速度方向。"""
    t1, t2 = theta
    s1, c1, s12, c12 = np.sin(t1), np.cos(t1), np.sin(t1 + t2), np.cos(t1 + t2)
    return np.array([[-L1 * s1 - L2 * s12, -L2 * s12], [L1 * c1 + L2 * c12, L2 * c12]])

**函数 `numeric_jacobian`**：用差分近似 J：每个关节挪 h，看手挪了多少，除以 h。

In [ ]:
def numeric_jacobian(theta: np.ndarray, h: float = 1e-6) -> np.ndarray:
    """用差分近似 J：每个关节挪 h，看手挪了多少，除以 h。"""
    cols = []
    for j in range(2):
        d = np.zeros(2)
        d[j] = h
        cols.append((fk(theta + d) - fk(theta - d)) / (2 * h))
    return np.stack(cols, axis=1)

**函数 `dls_ik`**：阻尼最小二乘 IK：Δθ = Jᵀ (J Jᵀ + λ² I)⁻¹ e，重复 iters 次。

In [ ]:
def dls_ik(target: np.ndarray, theta0: np.ndarray, lam: float = 0.05, iters: int = 100) -> np.ndarray:
    """阻尼最小二乘 IK：Δθ = Jᵀ (J Jᵀ + λ² I)⁻¹ e，重复 iters 次。"""
    theta = theta0.astype(float).copy()
    for _ in range(iters):
        e = target - fk(theta)
        J = jacobian(theta)
        theta = theta + J.T @ np.linalg.solve(J @ J.T + lam**2 * np.eye(2), e)
    return theta

**函数 `manipulability`**：可操作度 w = sqrt(det(J Jᵀ))，越接近 0 越靠近奇异。

In [ ]:
def manipulability(theta: np.ndarray) -> float:
    """可操作度 w = sqrt(det(J Jᵀ))，越接近 0 越靠近奇异。"""
    J = jacobian(theta)
    return float(np.sqrt(max(np.linalg.det(J @ J.T), 0.0)))

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
theta = np.radians([30.0, 60.0])
print("\n第一步：自由度——平面两连杆有 2 个关节，手的位置 (x, y) 也是 2 个数")
print("  三维空间放任意位姿要 6 个数：x, y, z + 3 个转角")

In [ ]:
print("\n第三步：正运动学 FK，θ1 = 30°，θ2 = 60°")
p = fk(theta)
print(f"  手在 x = {p[0]:.4f} m, y = {p[1]:.4f} m")
print(f"  工作空间：内圈半径 |L1 − L2| = {abs(L1 - L2):.1f} m，外圈 L1 + L2 = {L1 + L2:.1f} m")

In [ ]:
print("\n第四步：逆运动学 IK——同一个点有肘朝上、肘朝下两组解")
up = dls_ik(p, np.radians([0.0, 90.0]))
down = dls_ik(p, np.radians([90.0, -60.0]))
for name, th in (("解 1", up), ("解 2", down)):
    print(f"  {name}: θ = {np.round(np.degrees(th), 2).tolist()}°  → FK 回去 {np.round(fk(th), 4).tolist()}")

In [ ]:
print("\n第五步：雅可比 ẋ = J θ̇，解析式和差分对一下")
J = jacobian(theta)
print("  解析 J =\n", np.round(J, 4))
print("  差分 J =\n", np.round(numeric_jacobian(theta), 4))
F = np.array([0.0, -10.0])  # 手上挂 10 N 的重物（编的）
print(f"  静力学 τ = Jᵀ F，F = {F.tolist()} N → τ = {np.round(J.T @ F, 4).tolist()} N·m")

In [ ]:
print("\n第六步：奇异——det J = L1 L2 sin θ2，θ2 → 0 时手臂伸直")
for deg in (60.0, 20.0, 5.0, 0.0):
    th = np.radians([30.0, deg])
    print(f"  θ2 = {deg:>4}°: det J = {np.linalg.det(jacobian(th)):+.4f}  可操作度 w = {manipulability(th):.4f}")

## 3. 代码里才看得到的

- 代码里角度一律是弧度：`np.radians([30, 60])` 先换算，打印时再 `np.degrees` 换回来。
- 两组解用哪一组，取决于初始猜测 `theta0`：从肘朝上的猜测出发就收敛到肘朝上那组。数值 IK 自己不会"挑解"。
- 不用 `np.linalg.inv` 求逆，而是 `np.linalg.solve(A, e)` 直接解方程，更稳也更快。
- 第二列第二行是 0：θ = (30°, 60°) 时第二节连杆正好竖直，只转肘，手只会横着动。

## 4. 自己改一改

- 把 `lam` 改成 0（不加阻尼），再让目标落在外圈 0.9 m 之外，看迭代会不会乱跳。
- 把 θ2 从 5° 一路改到 −5°，看 det J 怎么经过 0 变号（肘从一侧翻到另一侧）。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 正解一路加角度，逆解要猜要迭代；雅可比管速度也管力。
> 手臂伸直就奇异：往外一毫米，关节转疯掉。